# 4. Aggregation

> Aggregate and combine the four cleaned datasets for downstream analysis and modelling.

## 1. Setup

In [21]:
import importlib
import sys, os

sys.path.insert(0, os.path.abspath(".."))

import src.code.io_utils as io

importlib.reload(io)

import pandas as pd
import numpy as np

## 2. Load Datasets

In [22]:
bdoss_clean    = io.load("../data/prepared/bdoss_clean.parquet")
crc_clean      = io.load("../data/prepared/crc_clean.parquet")
credscore_clean = io.load("../data/prepared/credscore_clean.parquet")
fama_clean     = io.load("../data/prepared/fama_clean.parquet")

[LOAD] ../data/prepared/bdoss_clean.parquet | shape: (2658187, 25)
[LOAD] ../data/prepared/crc_clean.parquet | shape: (3034749, 14)
[LOAD] ../data/prepared/credscore_clean.parquet | shape: (62873, 5)
[LOAD] ../data/prepared/fama_clean.parquet | shape: (141115, 10)


## 3. Aggregate BDOSS → one row per contract (CONTRIB × DOSSIER)

In [23]:
print("bdoss_clean:    ", bdoss_clean.shape)
print("crc_clean:      ", crc_clean.shape)
print("credscore_clean:", credscore_clean.shape)
print("fama_clean:     ", fama_clean.shape)

bdoss_clean:     (2658187, 25)
crc_clean:       (3034749, 14)
credscore_clean: (62873, 5)
fama_clean:      (141115, 10)


In [24]:
# Sort so that "last" aggregation = most recent observation per contract
bdoss_clean = bdoss_clean.sort_values(["CONTRIB", "DOSSIER", "OBS_DATE"])

In [25]:
# Derive POS flags and first event dates per contract (vectorized pivot)
# For each contract: was SOL/SAN/RBT ever observed, and when first?
pos_min = (
    bdoss_clean[bdoss_clean["POS"].isin(["SOL", "SAN", "RBT"])]
    .groupby(["CONTRIB", "DOSSIER", "POS"])["OBS_DATE"]
    .min()
    .reset_index()
    .pivot(index=["CONTRIB", "DOSSIER"], columns="POS", values="OBS_DATE")
    .reset_index()
)
pos_min.columns.name = None
pos_min = pos_min.rename(columns={"SOL": "OBS_DATE_SOL", "SAN": "OBS_DATE_SAN", "RBT": "OBS_DATE_RBT"})

# Binary flags: 1 if the contract ever had that POS status
for pos_val in ["SOL", "SAN", "RBT"]:
    pos_min[f"HAS_{pos_val}"] = pos_min[f"OBS_DATE_{pos_val}"].notna().astype(int)

pos_flags = pos_min[["CONTRIB", "DOSSIER",
                      "HAS_SOL", "HAS_SAN", "HAS_RBT",
                      "OBS_DATE_SOL", "OBS_DATE_SAN", "OBS_DATE_RBT"]]

In [26]:
drop_cols = ["OBS_DATE", "POS", "DCSP", "PTT"]
bdoss_agg_input = bdoss_clean.drop(columns=drop_cols)

agg_dict = {
    "DPOS":        "last",
    "DCREAT":      "last",
    "DATFIN":      "last",
    "D1FIN":       "last",
    "DURDEG":      "max",
    "RANGPRO":     "max",
    "RANGCLI":     "max",
    "MTFINO":      "max",
    "MTFIN":       "max",
    "MENSALIDADE": "median",
    "CRD":         "last",
    "SREC":        "last",
    "RN":          "max",
    "RD":          "max",
    "RISK":        "last",
    "RISKA":       "last",
    "RESSO":       "last",
    "CSP":         "last",
    "NBENF":       "last",
}

bdoss_aggregated = (
    bdoss_agg_input
    .groupby(["CONTRIB", "DOSSIER"], sort=False)
    .agg(agg_dict)
    .reset_index()
)

In [27]:
# Attach POS flags and event dates to the contract-level aggregation
bdoss_aggregated = bdoss_aggregated.merge(pos_flags, on=["CONTRIB", "DOSSIER"], how="left")

# Contracts with no SOL/SAN/RBT observations → fill missing flags with 0
for col in ["HAS_SOL", "HAS_SAN", "HAS_RBT"]:
    bdoss_aggregated[col] = bdoss_aggregated[col].fillna(0).astype(int)

In [28]:
print(f"Input rows  : {bdoss_clean.shape[0]:,}")
print(f"Output rows : {bdoss_aggregated.shape[0]:,}")
print(f"Columns     : {bdoss_aggregated.shape[1]}")

nulls = bdoss_aggregated.isnull().sum()
nulls = nulls[nulls > 0]
print(f"\nNull counts:\n{nulls if not nulls.empty else 'None'}")

bdoss_aggregated.head(3)

Input rows  : 2,658,187
Output rows : 185,600
Columns     : 27

Null counts:
OBS_DATE_SOL    157392
OBS_DATE_SAN    136799
OBS_DATE_RBT    182326
dtype: int64


,CONTRIB,DOSSIER,DPOS,DCREAT,DATFIN,D1FIN,DURDEG,RANGPRO,RANGCLI,MTFINO,...,RISKA,RESSO,CSP,NBENF,HAS_SOL,HAS_SAN,HAS_RBT,OBS_DATE_SOL,OBS_DATE_SAN,OBS_DATE_RBT
0,00008246f87bcc3c17b90629bb183fe2e58795176310f0...,36cc29a11081476fb7fcaed11aafef6fd441c3d172f4ff...,2024-10-03,2024-06-25,2024-06-27,2024-06-27,84.0,3.0,19.0,8000.0,...,0.0,1513.466,80.0,0.0,0,1,0,NaT,2024-09-30,NaT
1,00008246f87bcc3c17b90629bb183fe2e58795176310f0...,a34f4d89a43b924b483dc829a65ccba8b0f90e0afc2063...,2025-11-17,2024-09-13,2024-09-30,2024-09-30,120.0,13.0,19.0,16000.0,...,0.0,1513.466,80.0,0.0,0,1,0,NaT,2025-11-30,NaT
2,0000ab2116257783438c70ff85a3e98f2d4194ebe53434...,9e4d186f6f66f2da4b816cbc6f6e05e640caf5ded07956...,2018-04-16,2018-03-29,2018-04-16,2018-04-16,120.0,91.0,91.0,20000.0,...,0.0,1113.258,80.0,1.0,0,0,0,NaT,NaT,NaT


In [29]:
os.makedirs("../data/prepared", exist_ok=True)
io.save(bdoss_aggregated, "../data/prepared/bdoss_aggregated.parquet")

[SAVE] ../data/prepared/bdoss_aggregated.parquet | shape: (185600, 27)


---
## 4. Aggregate BDOSS → one row per customer (CONTRIB)

In [30]:
# Sort so "last" picks the most recent contract per customer
bdoss_aggregated = bdoss_aggregated.sort_values(["CONTRIB", "DCREAT"])

agg_dict = {
    "DOSSIER":       "count",
    "DCREAT":        ["min", "max"],
    "DPOS":          "max",
    "DATFIN":        "max",
    "D1FIN":         "min",
    "DURDEG":        ["min", "max", "median"],
    "RANGPRO":       ["min", "max", "median"],
    "RANGCLI":       ["min", "max", "median"],
    "MTFIN":         "sum",
    "MTFINO":        "sum",
    "MENSALIDADE":   "sum",
    "CRD":           "sum",
    "SREC":          "sum",
    "RN":            "sum",
    "RD":            "sum",
    "RISK":          "last",
    "RISKA":         "max",
    "RESSO":         ["min", "max", "median"],
    "CSP":           "last",
    "NBENF":         "last",
    "HAS_SOL":       ["max", "sum"],
    "HAS_SAN":       ["max", "sum"],
    "HAS_RBT":       ["max", "sum"],
    "OBS_DATE_SOL":  "max",
    "OBS_DATE_SAN":  "max",
    "OBS_DATE_RBT":  "max",
}

bdoss_customer = (
    bdoss_aggregated
    .groupby("CONTRIB", sort=False)
    .agg(agg_dict)
    .reset_index()
)

# Flatten MultiIndex columns → readable names
col_rename = {
    ("CONTRIB",      ""):         "CONTRIB",
    ("DOSSIER",      "count"):    "N_CONTRACTS",
    ("DCREAT",       "min"):      "FIRST_DCREAT",
    ("DCREAT",       "max"):      "LAST_DCREAT",
    ("DPOS",         "max"):      "LAST_DPOS",
    ("DATFIN",       "max"):      "LAST_DATFIN",
    ("D1FIN",        "min"):      "FIRST_D1FIN",
    ("DURDEG",       "min"):      "MIN_DURDEG",
    ("DURDEG",       "max"):      "MAX_DURDEG",
    ("DURDEG",       "median"):   "MEDIAN_DURDEG",
    ("RANGPRO",      "min"):      "MIN_RANGPRO",
    ("RANGPRO",      "max"):      "MAX_RANGPRO",
    ("RANGPRO",      "median"):   "MEDIAN_RANGPRO",
    ("RANGCLI",      "min"):      "MIN_RANGCLI",
    ("RANGCLI",      "max"):      "MAX_RANGCLI",
    ("RANGCLI",      "median"):   "MEDIAN_RANGCLI",
    ("MTFIN",        "sum"):      "TOTAL_MTFIN",
    ("MTFINO",       "sum"):      "TOTAL_MTFINO",
    ("MENSALIDADE",  "sum"):      "TOTAL_MENSALIDADE",
    ("CRD",          "sum"):      "TOTAL_CRD",
    ("SREC",         "sum"):      "TOTAL_SREC",
    ("RN",           "sum"):      "TOTAL_RN",
    ("RD",           "sum"):      "TOTAL_RD",
    ("RISK",         "last"):     "LAST_RISK",
    ("RISKA",        "max"):      "MAX_RISKA",
    ("RESSO",        "min"):      "MIN_RESSO",
    ("RESSO",        "max"):      "MAX_RESSO",
    ("RESSO",        "median"):   "MEDIAN_RESSO",
    ("CSP",          "last"):     "CSP",
    ("NBENF",        "last"):     "NBENF",
    ("HAS_SOL",      "max"):      "EVER_SOL",
    ("HAS_SOL",      "sum"):      "N_SOL",
    ("HAS_SAN",      "max"):      "EVER_SAN",
    ("HAS_SAN",      "sum"):      "N_SAN",
    ("HAS_RBT",      "max"):      "EVER_RBT",
    ("HAS_RBT",      "sum"):      "N_RBT",
    ("OBS_DATE_SOL", "max"):      "LAST_OBS_DATE_SOL",
    ("OBS_DATE_SAN", "max"):      "LAST_OBS_DATE_SAN",
    ("OBS_DATE_RBT", "max"):      "LAST_OBS_DATE_RBT",
}

bdoss_customer.columns = [col_rename.get(col, col[0] if isinstance(col, tuple) else col)
                          for col in bdoss_customer.columns]

In [31]:
# Validation
print(f"Input contracts : {bdoss_aggregated.shape[0]:,}")
print(f"Output customers: {bdoss_customer.shape[0]:,}")
print(f"Columns         : {bdoss_customer.shape[1]}")

assert bdoss_customer["CONTRIB"].nunique() == bdoss_customer.shape[0], "Duplicate CONTRIBs!"
assert bdoss_customer["N_CONTRACTS"].sum() == bdoss_aggregated.shape[0], "Row count mismatch!"

nulls = bdoss_customer.isnull().sum()
nulls = nulls[nulls > 0]
print(f"\nNull counts (expected only date cols for customers without SOL/SAN/RBT):\n{nulls if not nulls.empty else 'None'}")

print("\nAll checks passed.")
bdoss_customer.head(3)

Input contracts : 185,600
Output customers: 148,729
Columns         : 39

Null counts (expected only date cols for customers without SOL/SAN/RBT):
LAST_OBS_DATE_SOL    121467
LAST_OBS_DATE_SAN    108126
LAST_OBS_DATE_RBT    145601
dtype: int64

All checks passed.


,CONTRIB,N_CONTRACTS,FIRST_DCREAT,LAST_DCREAT,LAST_DPOS,LAST_DATFIN,FIRST_D1FIN,MIN_DURDEG,MAX_DURDEG,MEDIAN_DURDEG,...,NBENF,EVER_SOL,N_SOL,EVER_SAN,N_SAN,EVER_RBT,N_RBT,LAST_OBS_DATE_SOL,LAST_OBS_DATE_SAN,LAST_OBS_DATE_RBT
0,00008246f87bcc3c17b90629bb183fe2e58795176310f0...,2,2024-06-25,2024-09-13,2025-11-17,2024-09-30,2024-06-27,84.0,120.0,102.0,...,0.0,0,0,1,2,0,0,NaT,2025-11-30,NaT
1,0000ab2116257783438c70ff85a3e98f2d4194ebe53434...,1,2018-03-29,2018-03-29,2018-04-16,2018-04-16,2018-04-16,120.0,120.0,120.0,...,1.0,0,0,0,0,0,0,NaT,NaT,NaT
2,0000c74654405ec1da4dbdcd00b86e397954043965d98e...,2,2001-09-21,2001-09-27,2014-06-27,2001-10-04,2001-09-21,60.0,60.0,60.0,...,2.0,1,2,0,0,0,0,2024-03-31,NaT,NaT


In [32]:
os.makedirs("../data/prepared", exist_ok=True)
io.save(bdoss_customer, "../data/prepared/bdoss_customer.parquet")
print("Saved bdoss_customer.parquet")

[SAVE] ../data/prepared/bdoss_customer.parquet | shape: (148729, 39)
Saved bdoss_customer.parquet


---
## 5. Target Variable Engineering

### Objective 1 — Early Settlement (`IS_EARLY_SETTLER`)
A customer is an early settler if at least one of their contracts was ever in `SAN` or `RBT` status.

### Objective 2 — Churn (`IS_CHURN`)
A customer churns if they paid off a contract (`SOL`, `SAN`, or `RBT`) **and** did not open a new contract within 30 days of that payoff date.

In [33]:
# --- Objective 1: IS_EARLY_SETTLER ---
bdoss_customer["IS_EARLY_SETTLER"] = (
    (bdoss_customer["EVER_SAN"] == 1) | (bdoss_customer["EVER_RBT"] == 1)
).astype(int)

print("IS_EARLY_SETTLER distribution:")
print(bdoss_customer["IS_EARLY_SETTLER"].value_counts())
print(f"  Early settlers: {bdoss_customer['IS_EARLY_SETTLER'].sum():,} ({bdoss_customer['IS_EARLY_SETTLER'].mean()*100:.1f}%)")

# --- Objective 2: IS_CHURN ---
# Most recent contract-end event date (SOL, SAN, or RBT)
last_end = bdoss_customer[["LAST_OBS_DATE_SOL", "LAST_OBS_DATE_SAN", "LAST_OBS_DATE_RBT"]].max(axis=1)

# Did the customer ever have a contract end?
had_end_event = (bdoss_customer[["EVER_SOL", "EVER_SAN", "EVER_RBT"]].max(axis=1) == 1)

# Gap in days between the most recent contract end and the most recent contract start
gap_days = (bdoss_customer["LAST_DCREAT"] - last_end).dt.days

# Churn: had an end event AND (no new contract opened after it OR gap > 30 days)
no_new_contract = bdoss_customer["LAST_DCREAT"] <= last_end
late_new_contract = gap_days > 30

bdoss_customer["IS_CHURN"] = (had_end_event & (no_new_contract | late_new_contract)).astype(int)

print("\nIS_CHURN distribution:")
print(bdoss_customer["IS_CHURN"].value_counts())
print(f"  Churners: {bdoss_customer['IS_CHURN'].sum():,} ({bdoss_customer['IS_CHURN'].mean()*100:.1f}%)")

IS_EARLY_SETTLER distribution:
IS_EARLY_SETTLER
0    107354
1     41375
Name: count, dtype: int64
  Early settlers: 41,375 (27.8%)

IS_CHURN distribution:
IS_CHURN
0    82407
1    66322
Name: count, dtype: int64
  Churners: 66,322 (44.6%)


In [34]:
io.save(bdoss_customer, "../data/prepared/bdoss_customer.parquet")
print("Saved final bdoss_customer.parquet with target variables.")
print(f"Final shape: {bdoss_customer.shape}")

[SAVE] ../data/prepared/bdoss_customer.parquet | shape: (148729, 41)
Saved final bdoss_customer.parquet with target variables.
Final shape: (148729, 41)
